# Model Insights
This notebook houses all the work needed to generate the model insights displayed with the prediction tool itself. This includes: PDP's, SHAP values, Residual analysis.

In [1]:
!pip install shap

In [ ]:
# Imports for model insights generation (PDPs, residuals, SHAP)
import pandas as pd
import numpy as np
import pickle
from sklearn.inspection import partial_dependence
import os
import shap

In [ ]:
# Load the saved model pipeline and the raw train/test splits.
# X_train is needed for PDP background distribution; X_test is used for SHAP and residuals.
with open('../deployment/model_artifacts_002.pkl', 'rb') as f:
    artifacts = pickle.load(f)

best_model = artifacts['model']
train_df = pd.read_csv('../data/train_data.csv')
test_df = pd.read_csv('../data/test_data.csv')
X_train = train_df.drop(columns='Sold_Price')
X_test = test_df.drop(columns='Sold_Price')

## Partial dependency plots

In [ ]:
# Compute Partial Dependence Plots for continuous features.
# PDPs show the marginal effect of one feature on the predicted price,
# averaging out all other features — useful for communicating model behavior to end users.
#
# Only continuous features are included here. Target-encoded categorical features
# produce step-function PDPs that are harder to interpret visually.

features_to_plot = [
    'Mileage', 'car_age', 'Engine_Displacement_L', 'mileage_per_year', "Gears"
]
pdp_results = []

# mileage_per_year has NaNs for cars with unknown age; fill with median so
# scikit-learn can compute grid percentiles without raising an error.
X_train_pdp = X_train.copy()
X_train_pdp['mileage_per_year'] = X_train_pdp['mileage_per_year'].fillna(X_train_pdp['mileage_per_year'].median())

print("Calculating Partial Dependency Plots...")
for feature in features_to_plot:
    pdp = partial_dependence(best_model, X_train_pdp, features=[feature], grid_resolution=50)
    
    # scikit-learn changed the grid key name between versions; handle both
    grid_key = 'values' if 'values' in pdp else 'grid_values'
    grid_values = pdp[grid_key][0]
    
    # The pipeline predicts in log space; invert to plot dollar-denominated prices
    avg_predictions_dollars = np.expm1(pdp['average'][0])
    
    for val, pred in zip(grid_values, avg_predictions_dollars):
        pdp_results.append({
            'Feature': feature,
            'Feature_Value': val,
            'Predicted_Price': pred
        })

In [ ]:
# Export PDP results for the frontend visualization layer
pdp_df = pd.DataFrame(pdp_results)
pdp_df.to_csv('../data/frontend_data/pdp_data.csv', index=False)
print("Saved PDP data to ../data/frontend_data/pdp_data.csv")

## Residual Analysis

In [ ]:
# Build residual DataFrame on the test set for frontend visualization.
# Using the test set (not training set) ensures residuals reflect true out-of-sample error.
df_residuals = X_test.copy()

y_test_dollars = test_df['Sold_Price']

# Predict in log space, then invert to dollars for comparison against actual prices
y_pred_log = best_model.predict(X_test)
y_pred_dollars = np.expm1(y_pred_log)

df_residuals['Sold_Price'] = y_test_dollars.values
df_residuals['Predicted_Price'] = y_pred_dollars

df_residuals.head()

In [ ]:
# Reconstruct calendar Year from auction_year and car_age for display purposes.
# Year is not in the feature set directly (car_age is used instead to avoid
# collinearity with auction_year), so we back-calculate it here.
if 'Year' not in df_residuals.columns and 'auction_year' in df_residuals.columns and 'car_age' in df_residuals.columns:
    df_residuals['Year'] = df_residuals['auction_year'] - df_residuals['car_age']

# Trim to only the columns the frontend needs — keeps the CSV small
cols_to_keep = ['Make', 'Model', 'Year', 'Sold_Price', 'Predicted_Price']
available_cols = [c for c in cols_to_keep if c in df_residuals.columns]

os.makedirs('../data/frontend_data', exist_ok=True)
df_residuals[available_cols].to_csv('../data/frontend_data/residual_data.csv', index=False)

## Global SHAP values

In [ ]:
# Compute global SHAP feature importance on the test set.
# SHAP (SHapley Additive exPlanations) attributes each prediction to individual features
# using game-theory-based marginal contributions, making it more reliable than
# tree-internal feature importance (which can be biased toward high-cardinality features).
#
# TreeExplainer is used because it exploits XGBoost's tree structure for exact,
# fast SHAP values — much faster than KernelExplainer for tree-based models.

# Extract the preprocessor and XGBoost model from the pipeline separately,
# because TreeExplainer needs the raw model, not the full sklearn Pipeline.
preprocessor = best_model.named_steps['preprocessor']
xgb_model = best_model.named_steps['xgb']

X_test_transformed = preprocessor.transform(X_test)

# Strip sklearn's "step__" prefix from feature names for readable output
feature_names = [n.split('__')[-1] for n in preprocessor.get_feature_names_out()]

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_transformed)

# Mean absolute SHAP = average magnitude of each feature's contribution across all predictions
shap_importance = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).head(20)

shap_importance.to_csv('../data/frontend_data/shap_importance.csv', index=False)
print("Saved SHAP importance to ../data/frontend_data/shap_importance.csv")
print(shap_importance)